# 17 — Sensitivity figures

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from analysis.lib.stats import nadeau_bengio_test, bh_fdr
from analysis.lib.plotting import VARIABLE_LABELS, apply_manuscript_style

CORE_POOLED = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "pooled"
SENS_ATLAS  = PROJECT_ROOT / "outputs" / "predictive_modeling_sensitivity" / "atlas_resolution"
SENS_STREAM = PROJECT_ROOT / "outputs" / "predictive_modeling_sensitivity_streamlines"
FIG_DIR     = PROJECT_ROOT / "outputs" / "predictive_modeling_sensitivity" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert (CORE_POOLED / "stacking_scores.csv").exists(), \
    "run notebook 01 first (core pooled CV outputs missing)"
ref = pd.read_csv(CORE_POOLED / "stacking_scores.csv")
ref = ref[["Target", "Fold", "AUPRC"]].rename(columns={"AUPRC": "AUPRC_ref"})
print(f"reference: {len(ref)} fold rows, {ref['Target'].nunique()} targets")

In [ ]:
# Fig S8 — atlas resolution ΔAUPRC
def load_variant_scores(parent_dir):
    rows = []
    if not parent_dir.exists():
        return pd.DataFrame(columns=["Variant", "Target", "Fold", "AUPRC"])
    for subdir in sorted(parent_dir.iterdir()):
        if not subdir.is_dir():
            continue
        csv = subdir / "stacking_scores.csv"
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        if "Variant" not in df.columns:
            df["Variant"] = subdir.name
        rows.append(df[["Variant", "Target", "Fold", "AUPRC"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def compute_deltas(variant_df, ref_df):
    """Per-fold paired ΔAUPRC (variant − reference), mean ± SE per variant × target."""
    if variant_df.empty:
        return pd.DataFrame()
    merged = variant_df.merge(ref_df, on=["Target", "Fold"], how="inner")
    merged["delta"] = merged["AUPRC"] - merged["AUPRC_ref"]
    out = (merged.groupby(["Variant", "Target"])["delta"]
           .agg(mean_delta="mean", sd="std", n_paired="count").reset_index())
    out["se_delta"] = out["sd"] / np.sqrt(out["n_paired"])
    return out.drop(columns="sd")


def plot_sensitivity_panel(stats_df, variant_order, variant_labels, palette,
                           title, out_path):
    if stats_df.empty:
        print(f"[{title}] no data")
        return
    sns.set_theme(style="whitegrid", font_scale=1.1)
    targets_present = [t for t in VARIABLE_LABELS if t in stats_df["Target"].unique()]
    variants_present = [v for v in variant_order if v in stats_df["Variant"].unique()]
    if not targets_present or not variants_present:
        print(f"[{title}] nothing to plot")
        return

    n_targets = len(targets_present)
    n_variants = len(variants_present)
    group_w = 0.85
    band_w = group_w / n_variants
    offsets = np.linspace(-group_w/2 + band_w/2, group_w/2 - band_w/2, n_variants)

    fig, ax = plt.subplots(figsize=(max(11, 1.7 * n_targets), 6.5))
    for x_idx, tgt in enumerate(targets_present):
        for g_idx, variant in enumerate(variants_present):
            cx = x_idx + offsets[g_idx]
            row = stats_df[(stats_df["Target"] == tgt) & (stats_df["Variant"] == variant)]
            if row.empty:
                continue
            mean = float(row["mean_delta"].iloc[0])
            se = float(row["se_delta"].iloc[0]) if pd.notna(row["se_delta"].iloc[0]) else 0.0
            ax.bar(cx, mean, width=band_w*0.95, color=palette[variant],
                   edgecolor="black", linewidth=0.6,
                   label=variant_labels[variant] if x_idx == 0 else None)
            if se > 0:
                ax.errorbar(cx, mean, yerr=se, color="black", linewidth=0.8,
                            capsize=3, zorder=4)

    ax.axhline(0, color="black", linewidth=0.7)
    for x in range(1, n_targets):
        ax.axvline(x - 0.5, color="gray", linewidth=0.3, alpha=0.4)
    ax.set_xticks(range(n_targets))
    ax.set_xticklabels([VARIABLE_LABELS[t] for t in targets_present],
                       fontsize=10, rotation=0)
    ax.set_ylabel("Δ AUPRC vs reference\n(variant − full 4-view stacker)",
                  fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.legend(bbox_to_anchor=(0.5, -0.10), loc="upper center", ncol=3,
              title="Variant", frameon=True, fontsize=9)
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".pdf"), bbox_inches="tight")
    print(f"saved {out_path}")
    return fig


atlas_df = load_variant_scores(SENS_ATLAS)
atlas_stats = compute_deltas(atlas_df, ref)
print(atlas_stats.round(4))

ATLAS_ORDER = ["regimen_mid", "regimen_high"]
ATLAS_LABELS = {
    "regimen_mid":  "Mid-res\nSchaefer-400 + Tian-32\n+ Nettekoven-32 + xtract",
    "regimen_high": "High-res\nSchaefer-1000 + Tian-54\n+ Nettekoven-64 + xtract",
}
ATLAS_PALETTE = {"regimen_mid": "#d95f02", "regimen_high": "#7570b3"}
plot_sensitivity_panel(atlas_stats, ATLAS_ORDER, ATLAS_LABELS, ATLAS_PALETTE,
                       title="Atlas resolution sensitivity (Δ vs low-res core)",
                       out_path=FIG_DIR / "sensitivity_atlas_resolution.png")

In [ ]:
# Fig S9 — streamline-subsample violins
apply_manuscript_style()
STREAM_FIG_DIR = SENS_STREAM / "figures"
STREAM_FIG_DIR.mkdir(parents=True, exist_ok=True)

ARM_ORDER = ["sub10", "sub25", "sub50", "sub75", "full"]
ARM_LABEL = {
    "sub10": "10 % streamlines",
    "sub25": "25 % streamlines",
    "sub50": "50 % streamlines",
    "sub75": "75 % streamlines",
    "full":  "100 % streamlines (production)",
}
ARM_COLOR = {
    "sub10": "#FAEBC7", "sub25": "#F9CB9C", "sub50": "#EFA86E",
    "sub75": "#D77A5A", "full": "#A03E3E",
}

frames = []
for arm in ARM_ORDER:
    df = pd.read_csv(SENS_STREAM / f"stacking_scores_{arm}.csv")
    df["Arm"] = arm
    frames.append(df[["Target", "Fold", "AUPRC", "Arm"]])
pooled = pd.concat(frames, ignore_index=True)

targets_list = [t for t in VARIABLE_LABELS if t in pooled["Target"].unique()]
n_targets = len(targets_list)
n_arms = len(ARM_ORDER)

sns.set_theme(style="whitegrid", font_scale=1.3,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})
fig, ax = plt.subplots(figsize=(22, 8))
fig.subplots_adjust(left=0.055, right=0.985, top=0.88, bottom=0.24)
sns.violinplot(data=pooled, x="Target", y="AUPRC", hue="Arm",
               hue_order=ARM_ORDER, palette=ARM_COLOR, order=targets_list,
               linewidth=0, saturation=0.85, cut=0, inner=None, width=0.86, ax=ax)

group_w = 0.86
band_w = group_w / n_arms
offsets = np.linspace(-group_w/2 + band_w/2, group_w/2 - band_w/2, n_arms)
rng = np.random.RandomState(0)
for x_idx, t in enumerate(targets_list):
    for a_idx, arm in enumerate(ARM_ORDER):
        cx = x_idx + offsets[a_idx]
        vals = pooled.loc[(pooled["Target"] == t) & (pooled["Arm"] == arm),
                          "AUPRC"].values
        if len(vals):
            jx = rng.normal(loc=cx, scale=0.012, size=len(vals))
            ax.scatter(jx, vals, s=6, color="gray", alpha=0.30, zorder=2)
            ax.scatter(cx, np.nanmean(vals), color="#222", marker="D", s=48, zorder=4)

ax.set_xticks(range(n_targets))
ax.set_xticklabels([VARIABLE_LABELS[t] for t in targets_list],
                   fontsize=14, fontweight="bold", rotation=0)
for _yl in ax.get_yticklabels():
    _yl.set_fontsize(13)
    _yl.set_fontweight("bold")
ax.set_ylabel("AUPRC (per fold)", fontsize=14, fontweight="bold")
ax.set_xlabel("")

handles = [Patch(facecolor=ARM_COLOR[a], edgecolor="none", label=ARM_LABEL[a])
           for a in ARM_ORDER]
handles.append(Line2D([0], [0], marker="D", color="w", markerfacecolor="#222",
                      markersize=9, linewidth=0, label="Mean test AUPRC"))
ax.legend(handles=handles, bbox_to_anchor=(0.5, -0.14), loc="upper center",
          frameon=True, ncol=3, fontsize=12, prop={"weight": "bold", "size": 12})

out_png = STREAM_FIG_DIR / "slnm_subsample_sensitivity_violin.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_png}")

In [ ]:
# Sensitivity: log1p-transformed infarct volume — compute (cached).
# Re-runs pooled Baseline + 4-view stacker with stroke-total_infarct_volume_ml
# replaced by log1p(volume); same folds as notebook 01 (paired).
from joblib import Parallel, delayed

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.pipelines import process_fold
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import pooled_cv

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
LOGVOL_DIR = PROJECT_ROOT / "outputs" / "predictive_modeling_sensitivity" / "log_volume"
LOGVOL_FIG = LOGVOL_DIR / "figures"
LOGVOL_DIR.mkdir(parents=True, exist_ok=True)
LOGVOL_FIG.mkdir(parents=True, exist_ok=True)
CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
VOL_COL = "stroke-total_infarct_volume_ml"

if (LOGVOL_DIR / "stacking_scores.csv").exists():
    print("[skip] log-volume outputs exist")
    lv_baseline_df = pd.read_csv(LOGVOL_DIR / "baseline_scores.csv")
    lv_stacking_df = pd.read_csv(LOGVOL_DIR / "stacking_scores.csv")
else:
    assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"
    clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
    acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
    clinical = clinical_df.loc[acute_pool].copy()
    # log1p handles zero-volume entries; everything else unchanged.
    clinical[VOL_COL] = np.log1p(pd.to_numeric(clinical[VOL_COL], errors="coerce"))
    print(f"pool {len(acute_pool)}; {VOL_COL} -> log1p")

    lesion_acute = load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool)
    flnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(acute_pool)
    slnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool)
    X_demographics, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
    cv = pooled_cv(seed=42)

    baseline_rows, stacking_rows = [], []
    for target in identify_binary_targets(clinical):
        y_series = clinical[target]
        valid_mask = y_series.notna()
        y = y_series[valid_mask].values
        X_base = X_demographics[valid_mask.values]
        n_demo = X_base.shape[1]

        print(f"--- Baseline (log volume): {target} (N={len(y)}) ---")
        baseline_rows.extend(Parallel(n_jobs=-1)(
            delayed(process_fold)(i, train, test, X_base, y, "Baseline", target, n_demo)
            for i, (train, test) in enumerate(cv.split(X_base, y))))

        view_configs = [
            ("baseline",     X_base),
            ("lesion_acute", np.hstack([X_base, lesion_acute.loc[valid_mask].values])),
            ("flnm_acute",   np.hstack([X_base, flnm_acute.loc[valid_mask].values])),
            ("slnm_acute",   np.hstack([X_base, slnm_acute.loc[valid_mask].values])),
        ]
        print(f"--- Stacking (log volume): {target} ---")
        fold_results = Parallel(n_jobs=-1)(
            delayed(process_stacking_fold)(i, train, test, view_configs, y, n_demo)
            for i, (train, test) in enumerate(cv.split(X_base, y)))
        for r in fold_results:
            stacking_rows.append({
                "Target": target, "Model": "Stacking", "Modality": "meta",
                "Fold": r["Fold"], "AUPRC": r["AUPRC"], "AUPRC_train": r["AUPRC_train"],
                "AUROC": r.get("AUROC", float("nan")),
                "AUROC_train": r.get("AUROC_train", float("nan")),
                "meta_C": r["meta_C"]})

    lv_baseline_df = pd.DataFrame(baseline_rows)
    lv_stacking_df = pd.DataFrame(stacking_rows)
    lv_baseline_df.to_csv(LOGVOL_DIR / "baseline_scores.csv", index=False)
    lv_stacking_df.to_csv(LOGVOL_DIR / "stacking_scores.csv", index=False)
    print(f"saved to {LOGVOL_DIR}")

In [ ]:
# Log-volume NB tests: (a) log vs raw encoding (two-sided, paired folds vs
# notebook 01), (b) Stacking vs Baseline within the log run (one-sided).
ref_stacker = pd.read_csv(CORE_POOLED / "stacking_scores.csv")
ref_all = pd.read_csv(CORE_POOLED / "all_scores.csv")
ref_baseline = ref_all[(ref_all["Model"] == "Baseline")
                       & (ref_all["Modality"] == "base")]

rows = []
for model, log_df, ref_df in [("Baseline", lv_baseline_df, ref_baseline),
                              ("Stacking", lv_stacking_df, ref_stacker)]:
    for target in sorted(lv_stacking_df["Target"].unique()):
        lv = log_df[log_df["Target"] == target].set_index("Fold")["AUPRC"]
        rv = ref_df[ref_df["Target"] == target].set_index("Fold")["AUPRC"]
        common = lv.index.intersection(rv.index)
        if len(common) < 2:
            continue
        res = nadeau_bengio_test(lv.loc[common].values, rv.loc[common].values,
                                 n_splits=10, test_size=0.1, alternative="two-sided")
        rows.append({"Model": model, "Target": target,
                     "mean_log": float(lv.loc[common].mean()),
                     "mean_raw": float(rv.loc[common].mean()),
                     "mean_diff": float(res["mean_diff"]),
                     "t_stat": float(res["t_stat"]), "p": float(res["p"]),
                     "n_paired": int(res["n_paired"])})
nb_logvraw = pd.DataFrame(rows)
nb_logvraw["p_adj_BH"] = nb_logvraw.groupby("Target")["p"].transform(
    lambda x: bh_fdr(x.values))
nb_logvraw.to_csv(LOGVOL_DIR / "nb_significance.csv", index=False)
print(nb_logvraw.round(4).to_string(index=False))

In [ ]:
# Log-volume violin (supplement figure): Baseline vs Stacking per target,
# markers = > Majority / Permuted / Baseline (NB one-sided, BH-FDR within target)
from analysis.lib.plotting import (compute_nb_sig_table, get_sig_flags,
                                   plot_sig_markers, sig_legend_entries)

LV_MODELS = ["Baseline", "Stacking"]
LV_COLOR = {"Baseline": "#AEC6CF", "Stacking": "#9B7EBD"}

long_rows = []
for _, r in lv_baseline_df.iterrows():
    long_rows.append({"Target": r["Target"], "Group": "Baseline",
                      "Fold": r["Fold"], "AUPRC": r["AUPRC"]})
for _, r in lv_stacking_df.iterrows():
    long_rows.append({"Target": r["Target"], "Group": "Stacking",
                      "Fold": r["Fold"], "AUPRC": r["AUPRC"]})
log_long = pd.DataFrame(long_rows)

# Null references: Majority ignores features and Permuted is a label-scramble
# chance check, and notebook 01 ran both on identical folds (same seed / pool /
# targets), so the log-volume arms can be paired against the core nulls.
core_nulls = pd.read_csv(CORE_POOLED / "all_scores.csv")
for null_model in ("Majority", "Permuted"):
    nn = core_nulls[(core_nulls["Model"] == null_model)
                    & (core_nulls["Modality"] == "base")]
    log_long = pd.concat([log_long,
                          nn[["Target", "Fold", "AUPRC"]].assign(Group=null_model)],
                         ignore_index=True)

# NB one-sided tests, BH-FDR within target: Stacking vs Majority / Permuted /
# Baseline; Baseline vs Majority / Permuted.
nb_log = compute_nb_sig_table(log_long, n_splits=10, test_size=0.1,
                              focal_groups=["Stacking"],
                              reference_groups=["Majority", "Permuted", "Baseline"])
nb_log_base = compute_nb_sig_table(log_long, n_splits=10, test_size=0.1,
                                   focal_groups=["Baseline"],
                                   reference_groups=["Majority", "Permuted"])
nb_log = pd.concat([nb_log, nb_log_base], ignore_index=True)
nb_log.to_csv(LOGVOL_DIR / "nb_significance_vs_nulls_and_baseline.csv", index=False)
print(nb_log.round(4).to_string(index=False))

lv_targets = [t for t in VARIABLE_LABELS if t in log_long["Target"].unique()]
n_lv = len(lv_targets)
group_w = 0.7
band_w = group_w / len(LV_MODELS)
offsets = np.linspace(-group_w/2 + band_w/2, group_w/2 - band_w/2, len(LV_MODELS))

sns.set_theme(style="whitegrid", font_scale=1.3,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})
fig, ax = plt.subplots(figsize=(20, 7))
plot_df = log_long[log_long["Group"].isin(LV_MODELS)]
sns.violinplot(data=plot_df, x="Target", y="AUPRC", hue="Group",
               hue_order=LV_MODELS, palette=LV_COLOR, order=lv_targets,
               linewidth=0, saturation=0.7, cut=0, inner=None, width=0.7, ax=ax)

ax.set_ylim(0, 1.24)
sig_base_y = 1.02
sig_step = 0.045
rng_jitter = np.random.RandomState(0)
for x_idx, tname in enumerate(lv_targets):
    for g_idx, group in enumerate(LV_MODELS):
        cx = x_idx + offsets[g_idx]
        vals = plot_df.loc[(plot_df["Target"] == tname)
                           & (plot_df["Group"] == group), "AUPRC"].values
        if len(vals):
            jx = rng_jitter.normal(loc=cx, scale=0.015, size=len(vals))
            ax.scatter(jx, vals, s=8, color="gray", alpha=0.35, zorder=2)
            ax.scatter(cx, np.nanmean(vals), color="#222", marker="D", s=55, zorder=3)
            ax.text(cx, sig_base_y - 0.035, f"{np.nanmean(vals):.2f}",
                    ha="center", va="top", fontsize=13, fontweight="bold",
                    color="#222", zorder=5)
        if len(nb_log):
            sig_flags = get_sig_flags(nb_log, tname, group)
            plot_sig_markers(ax, cx, sig_flags, sig_base_y, sig_step,
                             marker_size=70, color=LV_COLOR[group])

for x in range(1, n_lv):
    ax.axvline(x - 0.5, color="gray", linewidth=0.3, alpha=0.4)
ax.set_xticks(range(n_lv))
ax.set_xticklabels([VARIABLE_LABELS[t] for t in lv_targets], fontsize=13, rotation=0)
ax.set_ylabel("Predictive performance (AUPRC)")
ax.set_xlabel("")
extra = [Line2D([0], [0], marker="D", color="w", markerfacecolor="#222",
                markersize=9, linewidth=0, label="Mean test AUPRC")]
extra += sig_legend_entries()
mh, ml = ax.get_legend_handles_labels()
ml = [{"Stacking": "Integrated"}.get(x, x) for x in ml]
ax.legend(mh + extra, ml + [h.get_label() for h in extra],
          bbox_to_anchor=(0.5, -0.10), loc="upper center", ncol=4,
          frameon=True, fontsize=12)
plt.tight_layout()
out_png = LOGVOL_FIG / "logvolume_pooled_violin.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_png}")